# CRISP-DM — Predicting `is_fraud` (shop.db / orders)

**Course:** IS 455 — Chapter 17 deployment assignment  
**Target:** Binary classification of fraudulent orders using the `orders` table.

This notebook follows CRISP-DM and aligns with Chapters 1–16 (business framing, EDA, preparation, classification, ensembles, evaluation/tuning, feature selection, serialization).

## 1. Business understanding

- **Problem:** Flag orders that are likely fraudulent before fulfillment so staff can verify high-risk orders first.
- **Success criteria:** Strong ranking quality (e.g. ROC-AUC / PR-AUC on held-out data), calibrated enough for a priority queue, and a serialized model usable by the deployed app.
- **Constraints:** Features must be available at order time (no leakage from future events).

## 2. Data understanding

Load from SQLite `shop.db`, profile `orders`, and relate to customers/products as needed for context.

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "shop.db"  # same file as the assignment
con = sqlite3.connect(DB_PATH)
orders = pd.read_sql_query("SELECT * FROM orders", con)
con.close()

orders.head()

In [ ]:
orders["is_fraud"].value_counts(normalize=True)

## 3. Data preparation & feature engineering

- Clean types, handle missing promo codes, encode categoricals.
- Engineer features such as billing vs shipping ZIP mismatch.
- Use `ColumnTransformer` / `Pipeline` (Chapter 7) so preparation is reproducible.

In [ ]:
# Example: reuse the same featurization as scripts/train_export_model.py
# (copy that file into a cell or %run it) so training matches production.

PAYMENT = ["bank", "card", "paypal", "crypto"]
DEVICE = ["desktop", "tablet", "mobile"]
COUNTRY = ["BR", "CA", "GB", "IN", "NG", "US"]

states = sorted(orders["shipping_state"].dropna().unique().tolist())

def featurize(df, states):
    names = (
        ["order_subtotal", "shipping_fee", "tax_amount", "order_total", "promo_used", "zip_mismatch"]
        + [f"payment_{p}" for p in PAYMENT]
        + [f"device_{d}" for d in DEVICE]
        + [f"country_{c}" for c in COUNTRY]
        + [f"state_{s}" for s in states]
    )
    rows = []
    for _, row in df.iterrows():
        z = 1.0 if str(row["billing_zip"]).strip() != str(row["shipping_zip"]).strip() else 0.0
        base = [
            float(row["order_subtotal"]), float(row["shipping_fee"]), float(row["tax_amount"]),
            float(row["order_total"]), float(row["promo_used"]), z,
        ]
        pm = str(row["payment_method"])
        base += [1.0 if pm == p else 0.0 for p in PAYMENT]
        dev = str(row["device_type"])
        base += [1.0 if dev == d else 0.0 for d in DEVICE]
        ic = str(row["ip_country"])
        base += [1.0 if ic == c else 0.0 for c in COUNTRY]
        st = str(row["shipping_state"])
        base += [1.0 if st == s else 0.0 for s in states]
        rows.append(base)
    import numpy as np
    return np.asarray(rows, dtype=float), names

X, feature_names = featurize(orders, states)
y = orders["is_fraud"].astype(int).values
X.shape, len(feature_names)

## 4. Modeling (classification + ensembles)

Train baseline + stronger models (e.g. logistic regression, random forest / gradient boosting from Chapter 13–14). Tune with cross-validation (Chapter 15).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])
log_reg.fit(X_train, y_train)

rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

## 5. Evaluation & feature importance (Ch. 15–16)

Compare models with ROC-AUC, precision/recall at a chosen threshold, and confusion matrices. Use permutation importance or RF feature importances for interpretation.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

for name, model in [("log_reg", log_reg), ("rf", rf)]:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = model.predict(X_test)
    print(name, "ROC-AUC", roc_auc_score(y_test, proba))
    print(classification_report(y_test, model.predict(X_test), digits=3))

## 6. Deployment — serialize for the pipeline (Ch. 17)

- Save the chosen model with **joblib** (`.sav` or `.joblib`).
- For the Vercel Next.js app, also export **`web/public/model.json`** using `scripts/train_export_model.py` so inference matches training.
- **Scheduled retraining:** Run the notebook or `python scripts/train_export_model.py` on a schedule (e.g. nightly on a CI runner or local machine), commit or upload the new `model.json`, then redeploy. Serverless hosts are not ideal for heavy training; keep training offline and deploy artifacts.

In [ ]:
import joblib

# .sav is the same joblib/pickle format as .joblib; common for Ch. 17 submissions.
joblib.dump(log_reg, "models/fraud_model.sav")
joblib.dump(rf, "models/fraud_model_rf.sav")
print("Saved models/fraud_model.sav and models/fraud_model_rf.sav")

### Next step (production parity)

From the repo root run:

```bash
pip install -r requirements-ml.txt
python scripts/train_export_model.py
```

That refreshes `web/public/model.json` (used by the deployed Next.js app) and `models/fraud_model.sav` (Python joblib — same trained pipeline as the JSON export).